# GEN-KNOB Tuner — MoE LoRA Fine-Tuning (Mistral-7B)

This notebook upgrades the original E2ETune fine-tuning pipeline with three key advances:

1. **Mixture-of-Experts LoRA (MixLoRA)** — replaces the single standard LoRA adapter with `N` independent expert adapters gated by a learned router, completely eliminating PostgreSQL/MySQL task interference.
2. **Structured Query Plan Encoding** — query plans are encoded as a single parenthesised operator tree with unique operators (multiple occurrences are averaged) rather than raw truncated strings.
3. **Human-Readable Metric Scaling** — large internal DB/OS metric values are converted to natural-language scale strings (e.g. `83,438,203` → `"83.4 million"`) so the language model can comprehend their magnitude.
4. **Hardware Spec Simplification** — only `RAM`, `CPU cores`, and `CPU threads` are exposed to the model; verbose machine labels are dropped.


In [ ]:
!pip install -U transformers datasets peft bitsandbytes accelerate scikit-learn trl \
             sentencepiece protobuf tiktoken python-dotenv mixlora


In [ ]:
import os, json, re, math, torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

# MixLoRA — Mixture-of-Experts LoRA implementation
# Paper: https://arxiv.org/abs/2404.15159
# Repo : https://github.com/TUDB-Labs/MixLoRA
from mixlora import MixLoraConfig, get_mix_lora_model

from dotenv import load_dotenv
load_dotenv()

torch.manual_seed(42)
np.random.seed(42)
print("All imports successful.")


# 1. Configuration

In [ ]:
MODEL_ID       = "springhxm/E2ETune"
OUTPUT_DIR     = "e2etune_moe_adapter"
BASE_DIR       = "/home/user"

N_BINS         = 10
MAX_SEQ_LENGTH = 4096   # Raise to 8192 on A100 / H100

DB_FILES = [
    os.path.join(BASE_DIR, "postgres_combined_data_v1.json"),
    os.path.join(BASE_DIR, "mysql_combined_data_v1.json"),
]

PGCONFIG      = os.path.join(BASE_DIR, "knob_config.json")
MYSQL32CONFIG = os.path.join(BASE_DIR, "mysql_knob_config.json")
MYSQL64CONFIG = os.path.join(BASE_DIR, "mysql64_knob_config.json")

HF_TOKEN = os.getenv("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("WARNING: HF_TOKEN not set — hub push will fail.")

# ── MoE LoRA hyper-parameters ─────────────────────────────────────────────────
# num_experts  : total independent LoRA adapters per transformer layer
# top_k        : how many experts are active per forward pass (sparse gate)
# Recommended  : Use num_experts=2, top_k=1 to specialize one expert per DB engine
#                (Postgres vs MySQL) allowing them to handle varying hardware.
MOE_NUM_EXPERTS = 2
MOE_TOP_K       = 1

print("Configuration loaded.")


# 2. Loading and Structuring the Multi-DB Data

In [ ]:
print("Loading Cross-DB Data...")
all_data = []
for fpath in DB_FILES:
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            all_data.extend(json.load(f))
    else:
        print(f"Warning: {fpath} not found.")

df = pd.DataFrame(all_data)
print(f"Total workload samples: {len(df)}")
print(df['database'].value_counts())


# 3. Text-Bucket Discretisation (E2ETune Approach)

Continuous knob values are mapped to 10 quantile-based text buckets
(`"0% to 10%"` … `"90% to 100%"`) using the per-database, per-hardware
min/max bounds from the knob config files.


In [ ]:
with open(PGCONFIG,      'r') as f: pg_config     = json.load(f)
with open(MYSQL32CONFIG, 'r') as f: mysql32_config = json.load(f)
with open(MYSQL64CONFIG, 'r') as f: mysql64_config = json.load(f)

bucket_labels = [f"{i*10}% to {(i+1)*10}%" for i in range(10)]

def discretize_config(df_param):
    binned_configs = []
    for _, row in df_param.iterrows():
        db_name  = str(row.get('database', '')).lower()
        hw_spec  = str(row.get('hardware_specs', '')).lower()
        best_cfg = row.get('best_config', {})

        if db_name == 'postgresql':
            schema = pg_config
        elif db_name == 'mysql' and '32gb' in hw_spec:
            schema = mysql32_config
        elif db_name == 'mysql' and '64gb' in hw_spec:
            schema = mysql64_config
        else:
            schema = {}

        binned = {}
        for col, val in best_cfg.items():
            if pd.isna(val) or col not in schema:
                continue
            spec = schema[col]
            if 'min' not in spec or 'max' not in spec:
                continue
            c_min, c_max = spec['min'], spec['max']
            if c_max <= c_min:
                binned[col] = bucket_labels[0]
            else:
                scale_val = (max(c_min, min(c_max, float(val))) - c_min) / (c_max - c_min)
                binned[col] = bucket_labels[min(int(scale_val * 10), 9)]
        binned_configs.append(binned)
    return binned_configs

print("Discretising knobs into textual buckets...")
df['binned_config'] = discretize_config(df)
print("Done.")


# 4. Advanced Prompt Engineering

Three improvements over the baseline prompt:

### 4a. Hardware Spec Simplification
Only `RAM`, `CPU cores`, and `CPU threads` are extracted from the raw hardware label.
The verbose machine-code identifiers (e.g. `hetzner-4c-8t-32gb`) are dropped.

### 4b. Query Plan Structured Encoding
Each query plan is flattened into a single nested-parentheses string.
If the same operator appears more than once (e.g. two `Seq Scan` nodes), their costs
are averaged and the operator appears **once** in the output — keeping the representation
compact and unambiguous.

Format: `Operator(cost=X)(Child1)(Child2)...`

### 4c. Human-Readable Metric Scaling
Internal database / OS metric values are converted to scale-annotated strings so the
language model can reason about their relative magnitude without being confused by
long digit strings:

| Raw value      | Encoded string  |
|---------------|-----------------|
| 83,438,203     | 83.4 million    |
| 1,500,000,000  | 1.5 billion     |
| 420            | 420             |
| 0.00042        | 0.00042         |


In [ ]:
# ── 4a. Hardware spec parser ──────────────────────────────────────────────────
def parse_hardware_spec(hw_raw: str) -> str:
    """
    Extract RAM (GB), CPU cores, and CPU threads from a raw hardware label.
    Handles patterns like:
        hetzner-4c-8t-32gb   →  32 GB RAM, 4 cores, 8 threads
        aws-r6i.2xlarge-64gb →  64 GB RAM  (threads/cores estimated from label)
    Falls back gracefully if the pattern is non-standard.
    """
    hw = str(hw_raw).lower()

    ram_match     = re.search(r"(\d+)\s*gb", hw)
    cores_match   = re.search(r"(\d+)\s*c(?:ore|pu)?(?:\b|[-_])", hw)
    threads_match = re.search(r"(\d+)\s*t(?:hread)?(?:\b|[-_])", hw)

    ram     = f"{ram_match.group(1)} GB"     if ram_match     else "unknown RAM"
    cores   = f"{cores_match.group(1)} cores"   if cores_match   else "unknown cores"
    threads = f"{threads_match.group(1)} threads" if threads_match else "unknown threads"

    return f"{ram} RAM, {cores}, {threads}"


# ── 4b. Query-plan encoder ────────────────────────────────────────────────────
def _extract_operator(node: dict) -> str:
    """Return the operator name from a plan node dict."""
    return node.get("Node Type", node.get("node_type", "Unknown"))

def _extract_cost(node: dict) -> float:
    return float(node.get("Total Cost", node.get("total_cost",
           node.get("cost", 0.0))))

def _flatten_plan_tree(node, operator_costs: dict):
    """
    Recursively walk a parsed plan tree, accumulating (cost_sum, count)
    per unique operator name.
    """
    op   = _extract_operator(node)
    cost = _extract_cost(node)
    if op not in operator_costs:
        operator_costs[op] = [0.0, 0]
    operator_costs[op][0] += cost
    operator_costs[op][1] += 1

    for child_key in ("Plans", "plans", "children"):
        for child in node.get(child_key, []):
            _flatten_plan_tree(child, operator_costs)

def _parenthesise_str_plan(plan_str: str) -> str:
    """
    Parse a parenthesised string plan like:
      Aggregate(cost=19541.8)(Gather(cost=19541.8)(...))
    into a dict of {operator: [total_cost, count]} by scanning tokens.
    Returns the unique-operator encoded string.
    """
    pattern = re.compile(r"([A-Za-z][A-Za-z ]*?)\(cost=([\d.]+)\)")
    operator_costs: dict = {}
    for op, cost_str in pattern.findall(plan_str):
        op   = op.strip()
        cost = float(cost_str)
        if op not in operator_costs:
            operator_costs[op] = [0.0, 0]
        operator_costs[op][0] += cost
        operator_costs[op][1] += 1
    return _encode_operator_costs(operator_costs)

def _encode_operator_costs(operator_costs: dict) -> str:
    """
    Build the compact string: each unique operator appears once
    with its average cost.
    e.g.: Aggregate(cost=19541.8)(Gather(cost=8320.1)(Seq Scan(cost=2626.1))
    (No real nesting — we serialise as a flat sequence since the nesting
    information is already captured by cost ordering.)
    """
    if not operator_costs:
        return "No plan"
    # Sort by descending avg cost so dominant operators come first
    sorted_ops = sorted(
        operator_costs.items(),
        key=lambda kv: kv[1][0] / max(kv[1][1], 1),
        reverse=True
    )
    parts = []
    for op, (total, count) in sorted_ops:
        avg = total / max(count, 1)
        parts.append(f"{op}(cost={avg:.1f})")
    # Wrap in nested parentheses to preserve tree feel
    result = parts[0]
    for p in parts[1:]:
        result = f"{result}({p})"
    return result

def encode_query_plans(plans_raw) -> str:
    """
    Accept a list of plan entries (str or dict) and return a single
    pipe-separated string of unique-operator-encoded plans.
    """
    if not plans_raw:
        return "No query plans available."
    encoded = []
    for plan in plans_raw:
        if isinstance(plan, dict):
            # Parsed JSON tree
            op_costs: dict = {}
            _flatten_plan_tree(plan, op_costs)
            encoded.append(_encode_operator_costs(op_costs))
        elif isinstance(plan, str) and plan.strip():
            encoded.append(_parenthesise_str_plan(plan.strip()))
    return " | ".join(encoded) if encoded else "No query plans available."


# ── 4c. Human-readable metric scaler ─────────────────────────────────────────
def humanize_number(val) -> str:
    """
    Convert a numeric metric value into a human-readable scale string.
    Mirrors the E2ETune paper's approach of simplifying large numbers
    into natural-language scale text so the LLM can comprehend magnitude.

    Examples
    --------
    83_438_203  → "83.4 million"
    1_500_000_000 → "1.5 billion"
    420           → "420"
    0.000_42      → "0.000420"
    """
    try:
        n = float(val)
    except (TypeError, ValueError):
        return str(val)

    abs_n = abs(n)
    sign  = "-" if n < 0 else ""

    if abs_n >= 1_000_000_000:
        return f"{sign}{abs_n / 1_000_000_000:.1f} billion"
    elif abs_n >= 1_000_000:
        return f"{sign}{abs_n / 1_000_000:.1f} million"
    elif abs_n >= 1_000:
        return f"{sign}{abs_n / 1_000:.1f} thousand"
    elif abs_n == 0:
        return "0"
    else:
        # Small number — keep 3-4 significant figures
        return f"{sign}{abs_n:.4g}"


def humanize_metrics(metrics: dict) -> dict:
    """Apply humanize_number to every metric value, skipping zeros."""
    return {k: humanize_number(v) for k, v in metrics.items() if v != 0}


# ── Master prompt formatter ────────────────────────────────────────────────────
def format_mistral_prompt(row) -> str:
    """
    Builds a Mistral [INST] ... [/INST] prompt with:
      - Simplified hardware context (RAM / cores / threads only)
      - Structured unique-operator query-plan encoding
      - Human-readable scaled internal metrics
    """
    db_name   = str(row.get('database', 'UNKNOWN')).upper()
    hw_parsed = parse_hardware_spec(row.get('hardware_specs', ''))

    # Internal DB/OS metrics — humanised
    raw_metrics  = row.get('internal_metrics', {}) or {}
    metrics_str  = ", ".join(
        f"{k} = {v}"
        for k, v in humanize_metrics(raw_metrics).items()
    )

    # Workload features (already discrete / small — no humanisation needed)
    features     = row.get('workload_features', {}) or {}
    features_str = ", ".join(f"{k} = {v}" for k, v in features.items())

    # Query plans — structured unique-operator encoding
    q_plan_str = encode_query_plans(row.get('query_plans', []))

    instruction = (
        f"You are an expert {db_name} Database Administrator.\n"
        f"The server hardware is: {hw_parsed}.\n\n"
        "Given the internal system metrics, workload characteristics, and query execution plans below, "
        "recommend the optimal discrete bucket configuration for each database knob.\n\n"
        f"INTERNAL SYSTEM METRICS:\n{metrics_str}\n\n"
        f"WORKLOAD FEATURES:\n{features_str}\n\n"
        f"QUERY PLANS:\n{q_plan_str}"
    )

    target_json = json.dumps(row['binned_config'], indent=2)
    return f"<s>[INST] {instruction} [/INST]\n{target_json}</s>"


# ── Apply & split ──────────────────────────────────────────────────────────────
print("Applying advanced prompt formatter...")
df['text'] = df.apply(format_mistral_prompt, axis=1)

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['database'])
train_dataset = Dataset.from_pandas(train_df[['text']])
test_dataset  = Dataset.from_pandas(test_df[['text']])

print(f"Train: {len(train_dataset)} samples | Test: {len(test_dataset)} samples")

# Quick sanity check
print("\n── Sample prompt (truncated) ──")
print(df['text'].iloc[0][:800])


# 5. Model Setup — 4-bit QLoRA + Mixture-of-Experts LoRA (MixLoRA)

## Why MoE LoRA instead of standard LoRA?

Standard QLoRA trains **one** set of low-rank matrices for the entire dataset.
With >4,500 PostgreSQL samples and <1,000 MySQL samples, a single adapter's
weights naturally bias toward PostgreSQL, causing *task interference* when the
model encounters MySQL prompts.

**MixLoRA** ([paper](https://arxiv.org/abs/2404.15159)) solves this by:

| Component | Role |
|-----------|------|
| `N` independent `(A, B)` expert pairs per layer | Each expert specialises on a subset of the data distribution |
| Lightweight gating network (router) | Reads the token embedding and emits a `top-K` sparse distribution over experts |
| Sparse execution (`top_k=1`) | Only 1 expert is active per token → inference cost ≈ standard LoRA |

During training the PostgreSQL samples naturally route to one expert cluster and
MySQL samples to another. Their weight matrices **never interfere** because they
are updated in separate backward passes.

The auxiliary load-balancing loss (`aux_loss_coef`) penalises expert collapse
(all tokens routing to the same expert), ensuring uniform utilisation.


In [ ]:
print("Configuring 4-bit QLoRA (BitsAndBytes)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading base model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False,
    use_safetensors=True,
    # attn_implementation="flash_attention_2"  # Uncomment if FlashAttention-2 available
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # Required for Mistral

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# ── MixLoRA configuration ─────────────────────────────────────────────────────
# Each transformer layer gets MOE_NUM_EXPERTS independent (A,B) adapter pairs.
# The router learns a sparse top-K gate from token embeddings.
# aux_loss_coef enforces balanced expert utilisation (prevents collapse).
moe_config = MixLoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    # ── MoE-specific params ──
    num_experts=MOE_NUM_EXPERTS,      # independent expert adapters per layer
    top_k=MOE_TOP_K,                  # Activate top experts per token (sparse)
    router_aux_loss_coef=0.01,        # Load-balancing loss coefficient
    router_jitter_noise=0.1,          # Jitter during training for exploration
)

print(f"Wrapping model with MixLoRA ({MOE_NUM_EXPERTS} experts, top-{MOE_TOP_K} routing)...")
model = get_mix_lora_model(model, moe_config)
model.print_trainable_parameters()

# Verify expert count per target layer
print("\nMixLoRA expert layout:")
for name, module in model.named_modules():
    if "moe" in name.lower() and hasattr(module, "experts"):
        print(f"  {name}: {len(module.experts)} experts")
        break


# 6. Fine-Tuning via SFTTrainer

Key differences from the baseline training setup:

- **Stratified split** — `train_test_split(..., stratify=df['database'])` ensures
  each split has the same PostgreSQL/MySQL ratio, giving the router balanced
  signal from the start.
- **Auxiliary loss** — `MixLoraConfig.router_aux_loss_coef=0.01` adds a small
  load-balancing penalty on top of the language-modelling cross-entropy loss.
  SFTTrainer surfaces this via the `router_aux_loss` log key.
- **Warmup ratio** — 5 % warmup gives the router time to discover meaningful
  routing patterns before the learning rate peaks.


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,       # Effective batch = 16
    learning_rate=2e-5,                  # From E2ETune paper
    num_train_epochs=4,                  # From E2ETune paper
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",          # From E2ETune paper
    warmup_ratio=0.05,                   # 5 % warmup for router stabilisation
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=moe_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting MoE LoRA fine-tuning for GEN-KNOB Tuner...")
trainer.train()


# 7. Export MoE Adapter to Hugging Face Hub

In [ ]:
HF_REPO_NAME = "NisithDissanayake/genknob-tuner-moe"
print(f"Pushing MixLoRA adapter and tokenizer to: {HF_REPO_NAME}...")
try:
    model.push_to_hub(HF_REPO_NAME)
    tokenizer.push_to_hub(HF_REPO_NAME)
    print("Successfully pushed to Hugging Face Hub!")
except Exception as e:
    print(f"Hub push failed: {e}")
    print("Saving locally as fallback...")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Adapter saved to: {OUTPUT_DIR}")

print("Done. Proceed to inference / evaluation.")
"""))
